In [0]:
import mlflow
# Disable autolog to prevent excessive logging during custom CV loops
mlflow.autolog(disable=True)

import importlib.util
import sys
from pyspark.ml import Pipeline
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# 1. LOAD CUSTOM MODULES (CV & Graph Features)
# -------------------------------------------------------------------------
# Load cv module directly from file path
cv_path = "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/notebooks/Cross Validator/cv.py"
spec = importlib.util.spec_from_file_location("cv", cv_path)
cv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cv)

# Load graph_features module
graph_features_path = "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/notebooks/Feature Engineering/graph_features.py"
spec = importlib.util.spec_from_file_location("graph_features", graph_features_path)
graph_features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(graph_features)


In [0]:
# Load data with graph & Meta features

data_loader_with_graph_and_meta = cv.FlightDelayDataLoader(suffix="_with_graph_and_metamodels")
data_loader_with_graph_and_meta.load() 


# Feature definitions
categorical_features = [
    'day_of_week', 'op_carrier', 'dep_time_blk', 'arr_time_blk', 'day_of_month', 'month'
]


numerical_features = [
    # ============================================================================
    # Flight Lineage Features (pre-applied in split.py)
    # ============================================================================
    'lineage_rank',
    'scheduled_lineage_rotation_time_minutes',
    'scheduled_lineage_turnover_time_minutes',
    'prev_flight_scheduled_flight_time_minutes',
    # Data Leakage Free Prior Flight Duration & Delay Features
    'safe_lineage_rotation_time_minutes',
    'safe_required_time_prev_flight_minutes',
    'safe_prev_departure_delay',  # NEW
    'safe_prev_arrival_delay',  # NEW
    'safe_time_since_prev_arrival', # NEW
    'prev_flight_distance',    # From top-performing XGBoost approach
    
    # # Safer but still risky
    # # Cumulative & Aggregated Features (exclude immediate previous flight - much safer!)
    # # These look at flights BEFORE the immediate previous flight (flights 1, 2, 3, etc., excluding n-1)
    # 'lineage_cumulative_delay',  # Sum of delays across flights before immediate previous flight
    # 'lineage_avg_delay_previous_flights',  # Average delay of flights before immediate previous flight
    # 'lineage_max_delay_previous_flights',  # Maximum delay of flights before immediate previous flight
    # 'lineage_num_previous_flights',  # Number of flights before immediate previous flight
    # 'lineage_expected_flight_time_minutes',  # Expected flight time based on historical patterns
    
    # ============================================================================
    # Meta-Model Predictions (pre-computed by add_meta_model_features.py)
    # ============================================================================
    'predicted_prev_flight_air_time_XGB_1', # Coming Soon!
    'predicted_prev_flight_turnover_time_XGB_1', # Coming Soon!
    'predicted_prev_flight_total_duration_XGB_1', # NEW!
    
    # ============================================================================
    # Weather Variables (Current Origin)
    # ============================================================================
    'hourlyprecipitation',
    'hourlysealevelpressure',
    'hourlyaltimetersetting',
    'hourlywetbulbtemperature',
    'hourlystationpressure',
    'hourlywinddirection',
    'hourlyrelativehumidity',
    'hourlywindspeed',
    'hourlydewpointtemperature',
    'hourlydrybulbtemperature',
    'hourlyvisibility',
    
    'crs_elapsed_time',        # Scheduled elapsed time
    'distance',                # Flight distance
    'elevation',               # Airport elevation (if available)


    # ============================================================================
    # Graph Features
    # ============================================================================

    'prev_flight_origin_pagerank_weighted', # New!
    'prev_flight_origin_pagerank_unweighted', # New!
    'origin_pagerank_weighted',
    'origin_pagerank_unweighted',
    'dest_pagerank_weighted',
    'dest_pagerank_unweighted'
]

In [0]:
# Impute missing values for numerical features
imputer = Imputer(
    inputCols=numerical_features,
    outputCols=[f"{c}_imputed" for c in numerical_features]
)

# StringIndexer and OneHotEncoder for categorical features
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe")
    for col in categorical_features
]

# Assemble all features
assembler = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in numerical_features] + [f"{c}_ohe" for c in categorical_features],
    outputCol="features"
)

# Random Forest Regressor
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="DEP_DELAY",  # Change to your label column if different
    predictionCol="prediction"
)

# Build pipeline
pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, rf])

In [0]:
cv_rf = cv.FlightDelayCV(
    estimator=pipeline,
    dataloader=data_loader_with_graph_and_meta,
    version="3M"
)

In [0]:
cv_rf.fit()

In [0]:
cv_rf = cv.FlightDelayCV(
    estimator=pipeline,
    dataloader=data_loader_with_graph_and_meta,
    version="60M"
)

In [0]:
cv_rf.fit()

In [0]:
cv_rf.evaluate()

In [0]:
# Random Forest Regressor
rf_50 = RandomForestRegressor(
    featuresCol="features",
    labelCol="DEP_DELAY",  # Change to your label column if different
    predictionCol="prediction",
    maxDepth=10,
    numTrees=50,
)

# Build pipeline
pipeline_50 = Pipeline(stages=[imputer] + indexers + encoders + [assembler, rf_100])

In [0]:
cv_rf = cv.FlightDelayCV(
    estimator=pipeline_50,
    dataloader=data_loader_with_graph_and_meta,
    version="60M"
)

In [0]:
cv_rf.fit()

In [0]:
cv_rf.evaluate()